In [2]:
%load_ext autoreload
%autoreload 2

In [1]:
# standard lib
import os, pwd, sys, json, yaml, atexit, tempfile, inspect
from pathlib import Path

# for data-science
import pandas as pd, numpy as np, quadfeather
from pyarrow import feather

# for plotting
import matplotlib as mpl, matplotlib.pyplot as plt, seaborn as sns

# for cellular-data
import scprep, anndata as ad

In [3]:
from featherplot.utils import MockSingleCellData, AnnDataProcessor, QuadFeatherRenamer
from featherplot.utils import SeriesToChannel

from featherplot.utils import collapse_user
from featherplot.deepscatter import Tileset

In [4]:
mocker = MockSingleCellData()
adata = mocker.adata

In [5]:
adata

AnnData object with n_obs × n_vars = 1000 × 100
    obs: 'barcodes', 'conditions'
    var: 'is_hvg'
    obsm: 'X_mock'
    layers: 'X_norm'

### Create Processor
> this will help us extract the embedding layer and the gene expression layer

In [6]:
pipe = AnnDataProcessor(adata, 'X_mock', 'X_norm')

#### sidecars

Deepscatter calls additional columns `sidecars`, in our case those are the columns of gene expression. We place these values in `df_s`.

In [9]:
df_s = pipe.get_sidecars()
df_s.head()

gene_symbols,gene_symbol 0,gene_symbol 1,gene_symbol 2,gene_symbol 3,gene_symbol 4,gene_symbol 5,gene_symbol 6,gene_symbol 7,gene_symbol 8,gene_symbol 9,...,gene_symbol 90,gene_symbol 91,gene_symbol 92,gene_symbol 93,gene_symbol 94,gene_symbol 95,gene_symbol 96,gene_symbol 97,gene_symbol 98,gene_symbol 99
barcodes,,,,,,,,,,,,,,,,,,,,,
barcode 0,0.803276,0.244524,-0.192386,-0.800793,-2.016473,0.821837,-0.298254,1.246355,0.357815,-0.528874,...,-1.088258,-2.642169,-0.956667,0.238179,-2.116511,-1.038880,0.282109,-1.040906,0.348024,0.703587
barcode 1,0.656503,-2.400139,1.251113,0.512456,0.306186,1.588643,-0.882513,0.265882,-1.937371,-0.279166,...,-0.395451,1.474536,0.433047,0.067309,1.097640,0.249324,0.538526,0.000884,2.739368,-1.101643
barcode 2,0.014030,1.228871,1.006785,0.720465,-0.121492,0.115829,0.695498,-0.896377,-0.402044,-1.376581,...,-0.416621,-0.667497,0.169813,0.311218,1.030311,0.396096,-0.754883,1.440937,0.849603,-0.541365
barcode 3,1.219619,0.834831,0.163721,0.343506,-0.164225,-0.095177,-1.137599,1.097777,-0.663735,0.797283,...,-1.010034,1.370871,-0.276237,-0.864229,-0.338752,0.555502,-1.301947,0.400086,0.205288,-1.246912
barcode 4,-0.355697,-0.515200,-1.566528,0.548676,0.168378,-1.152343,-0.225759,1.476776,-0.589473,0.030619,...,0.351259,0.823491,-0.408399,0.725561,-0.438170,-0.834018,-0.025542,1.571137,1.902323,-0.986374


In [ ]:
import pandas as pd


df = pd.read_csv(filepath_or_buffer="sample.csv")

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel

 
tokenizer = AutoTokenizer.from_pretrained("nomic-ai/nomic-embed-text-v1.5", trust_remote_code=True)
model = AutoModel.from_pretrained("nomic-ai/nomic-embed-text-v1.5", trust_remote_code=True)
 
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0]
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

 
texts = df["COMPLAINT_SUMMARY"].astype(str).tolist()

encoded_input = tokenizer(texts, padding=True, truncation=True, max_length=8192, return_tensors='pt')

 
with torch.no_grad():
    model_output = model(**encoded_input)
    
 
embeddings = mean_pooling(model_output, encoded_input["attention_mask"])
 
embeddings = F.normalize(embeddings, p=2, dim=1)
 
embeddings_hd = embeddings.cpu().numpy()
print("High-dimensional embeddings shape:", embeddings_hd.shape)
from sklearn.manifold import TSNE # change dimension reduction using UMAP

tsne = TSNE(n_components=2, random_state=42, perplexity=2, n_iter=1000)
embeddings_2d = tsne.fit_transform(embeddings_hd)
print("2D embeddings shape:", embeddings_2d.shape)

df["x"] = embeddings_2d[:, 0]
df["y"] = embeddings_2d[:, 1]
df["z"] = embeddings_2d[:, 2]

 
metadata_fields = [
    "COMPLAINT_ID", "CATEGORY", "SUBCATEGORY", "UPDATED_DATE", "STATE"
]

# Create df_q with selected fields + coordinates
df_q = df[metadata_fields].copy()
df_q["x"] = df["x"]
df_q["y"] = df["y"]
# df_q["z"] = df["z"]

# Convert categorical columns to strings (important for tiling)
df_q = df_q.astype(str)

# Save to quadfeather-compatible format
df_q.to_parquet("df_q.parquet")  # This is an alternative to quadfeather format


<All keys matched successfully>


High-dimensional embeddings shape: (60, 768)


/home/hm3/Desktop/complaint-plotter/.venv/lib/python3.13/site-packages/sklearn/manifold/_t_sne.py:1164: FutureWarning: 'n_iter' was renamed to 'max_iter' in version 1.5 and will be removed in 1.7.
  warnings.warn(


2D embeddings shape: (60, 3)
df_q prepared successfully:   COMPLAINT_ID  CATEGORY     SUBCATEGORY UPDATED_DATE STATE           x  \
0         C001   Billing      Overcharge   2025-01-15    CA  -40.754314   
1         C002   Service           Delay   2025-01-20    TX   20.613604   
2         C003   Product  Defective Item   2025-01-25    NY   58.592487   
3         C004   Support    Unresponsive   2025-01-30    FL  -17.683537   
4         C005  Delivery    Missing Item   2025-02-01    IL  -29.198523   

            y           z  
0   3.1160483   40.358833  
1  0.15664104   19.957281  
2  -26.991922  -1.0639364  
3   53.164185  -58.103943  
4  -23.397417  -20.732088  


In [8]:
df_q.head()

NameError: name 'df_q' is not defined

#### points

If our gene expression features are called `sidecars`, then what is the embedding layer called? Well it is just the "points" of the plot, so we will store these values in `df_p`.

**NOTE**: we also store conditions with `df_p` as whatever is in this DataFrame will be loaded by `Deepscatter` automatically. 

In [10]:
df_p = pipe.get_embedding()
df_p = df_p.join(pipe.adata.obs.conditions)
df_p.head()

,MOCK_1,MOCK_2,MOCK_3,conditions
barcodes,,,,
barcode 0,0.272783,0.277263,NaN,condition 0
barcode 1,-0.376359,-1.905994,NaN,condition 1
barcode 2,0.356305,0.017158,NaN,condition 2
barcode 3,0.134571,-0.100901,NaN,condition 3
barcode 4,0.856395,-2.030319,NaN,condition 0


#### Combined
Now we combine `df_p` (points + condition) with `df_s` ("sidecars" i.e. gene expression). This is necessary as for script later on where we need to add the sidecars to already the `quadfeather`-ed (tiled) point data. 

In [11]:
df_all = df_p.join(df_s)
df_all.head()

,MOCK_1,MOCK_2,MOCK_3,conditions,gene_symbol 0,gene_symbol 1,gene_symbol 2,gene_symbol 3,gene_symbol 4,gene_symbol 5,...,gene_symbol 90,gene_symbol 91,gene_symbol 92,gene_symbol 93,gene_symbol 94,gene_symbol 95,gene_symbol 96,gene_symbol 97,gene_symbol 98,gene_symbol 99
barcodes,,,,,,,,,,,,,,,,,,,,,
barcode 0,0.272783,0.277263,NaN,condition 0,0.803276,0.244524,-0.192386,-0.800793,-2.016473,0.821837,...,-1.088258,-2.642169,-0.956667,0.238179,-2.116511,-1.038880,0.282109,-1.040906,0.348024,0.703587
barcode 1,-0.376359,-1.905994,NaN,condition 1,0.656503,-2.400139,1.251113,0.512456,0.306186,1.588643,...,-0.395451,1.474536,0.433047,0.067309,1.097640,0.249324,0.538526,0.000884,2.739368,-1.101643
barcode 2,0.356305,0.017158,NaN,condition 2,0.014030,1.228871,1.006785,0.720465,-0.121492,0.115829,...,-0.416621,-0.667497,0.169813,0.311218,1.030311,0.396096,-0.754883,1.440937,0.849603,-0.541365
barcode 3,0.134571,-0.100901,NaN,condition 3,1.219619,0.834831,0.163721,0.343506,-0.164225,-0.095177,...,-1.010034,1.370871,-0.276237,-0.864229,-0.338752,0.555502,-1.301947,0.400086,0.205288,-1.246912
barcode 4,0.856395,-2.030319,NaN,condition 0,-0.355697,-0.515200,-1.566528,0.548676,0.168378,-1.152343,...,0.351259,0.823491,-0.408399,0.725561,-0.438170,-0.834018,-0.025542,1.571137,1.902323,-0.986374


### QuadFeatherRenamer

Note: `quadfeather` and `deepscatter` are both under active development so things change all the time. At the moment `quadfeather` requires that `x` and `y` be in your DataFrame (it doesn't mind if `z` is there too). So this will handle the renaming of our columns.

In [12]:
qfr = QuadFeatherRenamer(df_all)

In [13]:
df_q, renamed = qfr.rename()
renamed

{'MOCK_1': 'x', 'MOCK_2': 'y', 'MOCK_3': 'z'}

In [14]:
from dataclasses import dataclass, field
from typing import Optional, List, Tuple, Union, Any, Dict
import pandas as pd

@dataclass
class DataFrameToMetadata:
    df: pd.DataFrame
    sidecars: Optional[List[str]] = field(default_factory=list)
    embedding: Optional[List[str]] = field(default_factory=lambda: ['x', 'y', 'z'])
    alt_names: Optional[Dict[str, str]] = field(default_factory=dict)
    include_index: Optional[bool] = True

    def __post_init__(self):
        if not self.sidecars:
            self.sidecars = self._default_sidecars()

        if not self.embedding:
            self.embedding = ['x', 'y', 'z']
        elif not all(e in self.embedding for e in ['x', 'y', 'z']):
            raise ValueError('Must have x, y, z in embedding')

    def _default_sidecars(self) -> List[str]:
        return sorted(set(self.df.columns) - set(self.embedding))

    def _do_one(self, name: str, col: pd.Series, is_index: bool = False) -> Tuple[Union[Any, None], Union[Dict[str, Any], None]]:
        success = None
        failure = None

        is_sidecar = name in self.sidecars and name not in self.embedding
        alt_name = self.alt_names.get(name)

        try:
            s2c = SeriesToChannel(col, is_sidecar, alt_name)
            channel = s2c.convert()

            # Ensure categorical type if applicable
            if not is_index and self.df.index.name != name and s2c.is_category():
                channel = s2c.as_category().convert()
                self.df.loc[:, name] = s2c.series.values

            success = channel
        except Exception as e:
            failure = {"name": name, "is_sidecar": is_sidecar, "alt_name": alt_name, "error": e}

        return success, failure

    def convert(self) -> Tuple[dict, List[Dict[str, Dict[str, Any]]]]:
        failed = []
        channels = {}

        if self.include_index:
            name = "index" if self.df.index.name is None else self.df.index.name
            succ, fail = self._do_one(name, pd.Series(self.df.index), is_index=True)
            if succ is not None:
                channels[name] = succ
            else:
                failed.append(fail)

        for name, col in self.df.items():  # Updated from iteritems() to items()
            succ, fail = self._do_one(name, col)
            if succ is not None:
                if name not in channels:
                    channels[name] = succ
            else:
                failed.append(fail)

        return channels, failed

    def to_dict(self) -> dict:
        channels, _ = self.convert()
        return {
            "index": self.df.index.name if self.df.index.name else "index",
            "n_points": len(self.df),
            "embedding": self.embedding,
            "sidecars": self.sidecars,
            "columns_metadata": {k: v.to_dict() for k, v in channels.items()},
            "tiles_dir": None,
        }

    def to_meta(self) -> dict:
        channels, _ = self.convert()
        return {
            "index": self.df.index.name if self.df.index.name else "index",
            "n_points": len(self.df),
            "embedding": self.embedding,
            "sidecars": self.sidecars,
            "columns_metadata": {k: v.to_meta() for k, v in channels.items()},
            "tiles_dir": None,
        }


### DataFrameToMetadata
 
`Deepscatter` is a really nice library; however, it also prefers to have its `plotAPI` method called with as much information as possible. This is a bit of a shame as it means that one you load your data with `deepscatter` you can't compute derived properties (e.g. domain of your data to scale the plot, check for what sidecars are availble, etc). 

The solution to this is simple. In order to have this information availble to us, we will just calculate it now (including which columns were renamed) and store it as metadata to use later

In [15]:
d2m = DataFrameToMetadata(
    df_q, 
    include_index=True,
    embedding='x y z conditions'.split(),
    alt_names={v:k for k,v in renamed.items()}
)

In [16]:
succ, fail = d2m.convert()
len(succ), len(fail)

(105, 0)

In [17]:
meta = d2m.to_meta()

print(meta)

{'index': 'barcodes', 'n_points': 1000, 'embedding': ['x', 'y', 'z', 'conditions'], 'sidecars': ['gene_symbol 0', 'gene_symbol 1', 'gene_symbol 10', 'gene_symbol 11', 'gene_symbol 12', 'gene_symbol 13', 'gene_symbol 14', 'gene_symbol 15', 'gene_symbol 16', 'gene_symbol 17', 'gene_symbol 18', 'gene_symbol 19', 'gene_symbol 2', 'gene_symbol 20', 'gene_symbol 21', 'gene_symbol 22', 'gene_symbol 23', 'gene_symbol 24', 'gene_symbol 25', 'gene_symbol 26', 'gene_symbol 27', 'gene_symbol 28', 'gene_symbol 29', 'gene_symbol 3', 'gene_symbol 30', 'gene_symbol 31', 'gene_symbol 32', 'gene_symbol 33', 'gene_symbol 34', 'gene_symbol 35', 'gene_symbol 36', 'gene_symbol 37', 'gene_symbol 38', 'gene_symbol 39', 'gene_symbol 4', 'gene_symbol 40', 'gene_symbol 41', 'gene_symbol 42', 'gene_symbol 43', 'gene_symbol 44', 'gene_symbol 45', 'gene_symbol 46', 'gene_symbol 47', 'gene_symbol 48', 'gene_symbol 49', 'gene_symbol 5', 'gene_symbol 50', 'gene_symbol 51', 'gene_symbol 52', 'gene_symbol 53', 'gene_sym

## Quadfeather Workflow

Now we can now run through thte `quadfeather` workflow right here in the notebook.

### 0) setup

In [18]:
# dump everything to downloads for easy access
outdir = os.path.expanduser('~/Downloads/featherplot')
qf_dir = os.path.join(outdir, 'tiles')
if not os.path.isdir(qf_dir):
    os.makedirs(qf_dir)


p_file = os.path.join(outdir, 'points.parquet')
# NOTE: we never use s_file
# s_file = os.path.join(outdir, 'extras.parquet')
f_file = os.path.join(qf_dir, 'sidecars.feather')
m_file = os.path.join(outdir, 'meta.yml')


tile_size = 1000

### 1) create tiles

In [19]:
d2m.df.drop(columns=df_s.columns).to_parquet(p_file)
# d2m.df.drop(columns=d2m.embedding).to_parquet(s_file)

In [20]:
!quadfeather --files {p_file} \
             --tile_size {tile_size} \
             --destination {qf_dir}

### 2) make single file

In [21]:
feather.write_feather(d2m.df.drop(columns=d2m.embedding), f_file)

### 3) run `add_sidecars.py`

In [24]:
tileset = Tileset(Path(qf_dir))
print(f_file)
# tileset.add_sidecars(f_file, d2m.df.index.name)

/home/hm3/Downloads/featherplot/tiles/sidecars.feather


note we copied `add_sidecars.py` so you can use it directly from this library

In [25]:
!featherplot add-sidecars --tileset {qf_dir}\
                         --sidecar {f_file} --key {d2m.df.index.name};

╭───────────────────── Traceback (most recent call last) ──────────────────────╮
│ /home/hm3/Desktop/complaint-plotter/.venv/lib/python3.13/site-packages/feath │
│ erplot/commands.py:37 in add_sidecars                                        │
│                                                                              │
│   34 │   verbose:bool = typer.Option(False, '--verbose', '-v', help='Print v │
│   35 ):                                                                      │
│   36 │   tileset = Tileset(tileset)                                          │
│ ❱ 37 │   tileset.add_sidecars(sidecar, key, verbose=verbose)                 │
│   38                                                                         │
│   39 # %% ../nbs/08_commands.ipynb 7                                         │
│   40 @app.command()                                                          │
│                                                                              │
│ ╭─────────────────────────

```sh
featherplot-py featherplot add-sidecars --help

Usage: featherplot add-sidecars 
[OPTIONS]
--tileset          PATH  Path to the tileset to add sidecars to.
--sidecar          PATH  Path to the new data to add to the tileset.
--key              TEXT  key to use for joining; must exist in both tables
--verbose  -v            Print verbose output.
--help                   Show this message and exit.
```

alternatively you can run the script form wherever you saved it

In [26]:
!python3 add_sidecars.py --tileset {qf_dir}\
                         --sidecar {f_file} --key {d2m.df.index.name};

python3: can't open file '/home/hm3/Desktop/complaint-plotter/add_sidecars.py': [Errno 2] No such file or directory


### 4) update metadata with directory

In [27]:
meta.keys()

dict_keys(['index', 'n_points', 'embedding', 'sidecars', 'columns_metadata', 'tiles_dir'])

In [28]:
# relative path to tiles
meta['tiles_dir'] = qf_dir.replace(outdir, '')
# full path to tiles
meta['full_path'] = collapse_user(qf_dir)

In [29]:
with open(m_file, 'w') as f:
    f.write(yaml.dump(meta))
    
print(f"Write out at {m_file}")

Write out at /home/hm3/Downloads/featherplot/meta.yml


### 5) cleanup